# Gold Layer - Customer Dimension

## Overview
This notebook generates the **customer dimension table** for the gold layer dimensional model.

## Purpose
Extends the star schema with customer demographics and loyalty information:
- Extracts unique customers from sales transactions
- Enriches with synthetic demographic data
- Enables customer segmentation and analysis

## Generated Table
- **sales.gold.customers_dim** - Customer dimension with demographics and loyalty tiers
  - customer_id (links to sales_fact)
  - first_name, last_name
  - email, phone
  - age, city
  - registration_date
  - loyalty_tier (Platinum, Gold, Silver, Bronze)
  - is_active

## Source Data
Reads customer IDs from: `sales.gold.sales_fact`

## Integration
Complements existing gold layer tables:
- sales.gold.sales_fact (fact table)
- sales.gold.category_dim (product categories)
- sales.gold.payment_dim (payment methods/locations)
- **sales.gold.customers_dim** (this table - customer demographics)

In [0]:
# ============================================================================
# Setup: Import Libraries
# ============================================================================

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType
from datetime import date, timedelta
import random

In [0]:
# ============================================================================
# Step 1: Read Sales Fact Table
# ============================================================================
# Load the fact table to extract unique customer IDs for dimension generation

sales_fact_df = spark.read.table("sales.gold.sales_fact")

In [0]:
# ============================================================================
# Step 2: Generate Customer Dimension Table
# ============================================================================
# Create customer dimension with synthetic demographic data
# Links to sales_fact via customer_id
# ============================================================================

# Sample data for realistic customer generation
first_names = ['Emma', 'Liam', 'Olivia', 'Noah', 'Ava', 'Ethan', 'Sophia', 'Mason', 'Isabella', 'William',
               'Mia', 'James', 'Charlotte', 'Benjamin', 'Amelia', 'Lucas', 'Harper', 'Henry', 'Evelyn']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez',
              'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore']
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose']
loyalty_tiers = ['Platinum', 'Gold', 'Silver', 'Bronze']

# Extract unique customer IDs from fact table
customer_ids = sales_fact_df.select('customer_id').distinct().collect()
customer_ids = [row['customer_id'] for row in customer_ids]

# Generate synthetic customer data
customers_data = []
for customer_id in customer_ids:
    first_name = random.choice(first_names)
    last_name = random.choice(last_names)
    email = f"{first_name.lower()}.{last_name.lower()}@email.com"
    phone = f"({random.randint(200, 999)}) {random.randint(100, 999)}-{random.randint(1000, 9999)}"
    age = random.randint(18, 70)
    city = random.choice(cities)
    registration_date = date(2020, 1, 1) + timedelta(days=random.randint(0, 1095))
    loyalty_tier = random.choice(loyalty_tiers)
    is_active = random.choice([True, True, True, False])  # 75% active
    
    customers_data.append(Row(
        customer_id=customer_id,
        first_name=first_name,
        last_name=last_name,
        email=email,
        phone=phone,
        age=age,
        city=city,
        registration_date=registration_date,
        loyalty_tier=loyalty_tier,
        is_active=is_active
    ))

# Create DataFrame
customers_dim_df = spark.createDataFrame(customers_data)

# Write to Delta table in gold layer
customers_dim_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales.gold.customers_dim")

print(f"✓ Created customer dimension with {customers_dim_df.count()} customers")
display(customers_dim_df)